# Análisis de la lista de artículos y Reglas de Asociación

Hasta ahora hemos clasificado y limpiado datos. Hoy vamos a construir el motor que genera miles de millones de dólares para Amazon, Spotify y Netflix: El Sistema de Recomendaciones.

In [2]:
import pandas as pd

# Simulamos 5 tickets de compra de la cafetería
datos = {
    'Frappe': [True, False, True, True, False],
    'Muffin': [True, False, True, False, False],
    'Agua':   [True, True, True, True, True]
}
df_tickets = pd.DataFrame(datos)
total_tickets = len(df_tickets)

print(f"Total de transacciones (N): {total_tickets}")
display(df_tickets)

Total de transacciones (N): 5


,Frappe,Muffin,Agua
0,True,True,True
1,False,False,True
2,True,True,True
3,True,False,True
4,False,False,True


## 1. Soporte (Support): La métrica de popularidad

La Fórmula: $Soporte(A \rightarrow B) = \frac{Transacciones\ que\ contienen\ A\ y\ B}{Total\ de\ Transacciones}$

El Concepto: ¿Qué tan frecuente es ver estos dos productos juntos en el carrito frente a todo el volumen de ventas? Sirve para descartar combinaciones raras.

Ejemplo de la vida real (Retail Regional): Si analizamos 10,000 tickets de compra de un Calimax durante un fin de semana en Baja California, y en 2,000 de esos tickets aparecen juntos la Carne para Asar y el Carbón, el soporte de esa regla es del 20%. Es una combinación con peso estadístico real para justificar mover los pasillos físicos, no un evento aislado.

¿Qué pregunta responde? "¿Vale la pena que gaste tiempo y dinero analizando esta combinación, o es un evento aislado que casi nadie compra?"

¿Qué tan común es que se venda un Frappe y un Muffin juntos frente a todas las ventas del día? Si el Soporte es del 1%, la regla puede ser cierta, pero no nos generará impacto financiero real. Solo nos importan combinaciones que muevan la aguja del negocio.

In [3]:
# 1. Contamos en cuántos tickets aparecen ambos productos al mismo tiempo
filtro_ambos = (df_tickets['Frappe'] == True) & (df_tickets['Muffin'] == True)
frecuencia_frappe_muffin = len(df_tickets[filtro_ambos])

# 2. Dividimos entre el universo total de tickets
soporte_ambos = frecuencia_frappe_muffin / total_tickets

print(f"Tickets con Frappe Y Muffin: {frecuencia_frappe_muffin}")
print(f"Soporte (Frappe y Muffin): {frecuencia_frappe_muffin} / {total_tickets} = {soporte_ambos} (o {soporte_ambos*100}%)")

Tickets con Frappe Y Muffin: 2
Soporte (Frappe y Muffin): 2 / 5 = 0.4 (o 40.0%)


## 2. Confianza (Confidence): La métrica de probabilidad condicionadaLa 

Fórmula: $Confianza(A \rightarrow B) = \frac{Soporte(A \cup B)}{Soporte(A)}$

El Concepto: Si el cliente ya tiene el producto A en la mano, ¿cuál es la probabilidad matemática de que también se lleve el producto B?

Ejemplo de la vida real (Streaming): Un usuario de Netflix le da play a la película "Shrek" (A). El algoritmo sabe que el 85% de las veces que alguien ve la primera película, termina viendo "Shrek 2" (B). La confianza de la regla (Shrek $\rightarrow$ Shrek 2) es 0.85. Por eso Netflix coloca la secuela inmediatamente en la pantalla de inicio al terminar los créditos. Es una apuesta casi segura.

¿Qué pregunta responde? "Si el cliente ya tiene el Producto A en las manos, ¿qué tanta seguridad estadística tengo de que aceptará llevarse el Producto B si se lo ofrezco?"

Si veo que el alumno ya pidió su Frappe, ¿qué probabilidad hay de que me diga que sí al Muffin? Nos dice qué tan confiable es la regla de $A \rightarrow B$, convirtiéndose en el motor principal de los sistemas de recomendación ("Los clientes que vieron X también compraron Y").

In [4]:
# 1. Contamos en cuántos tickets aparece el antecedente (Frappe) de forma aislada
frecuencia_frappe = len(df_tickets[df_tickets['Frappe'] == True])

# 2. Dividimos la frecuencia conjunta (calculada arriba) entre la frecuencia del antecedente
confianza_regla = frecuencia_frappe_muffin / frecuencia_frappe

print(f"Tickets totales que incluyen Frappe: {frecuencia_frappe}")
print(f"Confianza (Frappe -> Muffin): {frecuencia_frappe_muffin} / {frecuencia_frappe} = {confianza_regla:.2f}")
print(f"Interpretación: Si alguien compra un Frappe, hay un {confianza_regla*100}% de probabilidad de que pida el Muffin.")

Tickets totales que incluyen Frappe: 3
Confianza (Frappe -> Muffin): 2 / 3 = 0.67
Interpretación: Si alguien compra un Frappe, hay un 66.66666666666666% de probabilidad de que pida el Muffin.


## Lift (Elevación): La métrica del impulso realLa 

Fórmula: $Lift = \frac{Confianza(A \rightarrow B)}{Soporte(B)}$

El Concepto: La métrica definitiva. Mide si el producto A realmente "impulsa" la venta del B, o si B simplemente es tan popular que se vende con todo (como el pan blanco).

Ejemplos de la vida real (Los 3 escenarios):
* Lift > 1 (Fuerte asociación - A impulsa a B): Comprar Fresas eleva drásticamente la probabilidad de comprar Crema Batida. El Lift será alto (ej. 4.5) porque muy poca gente entra a comprar crema batida sola; su venta depende casi por completo de las fresas.
* Lift = 1 (Eventos Independientes - La coincidencia): Comprar Detergente y comprar Agua embotellada. El Lift es cercano a 1.0. Ambos productos se venden muchísimo todo el tiempo. Que estén juntos en el ticket es una simple coincidencia estadística; ponerlos en promoción juntos sería desperdiciar dinero.
* Lift < 1 (Asociación Negativa - A repele a B): Comprar un Paraguas y comprar Bloqueador Solar. Tienen un Lift de 0.2. Si un cliente compra uno, las condiciones ambientales dictan que activamente evitará el otro. Nunca armarías un paquete promocional cruzando estos artículos.

¿Qué pregunta responde? "¿La compra del Producto A realmente motivó al cliente a comprar el Producto B, o simplemente se vendieron juntos por coincidencia porque el B es súper popular?"

¿El alumno compró el Muffin gracias a que compró el Frappe, o lo compró simplemente porque los Muffins se venden muchísimo de manera natural? El Lift desenmascara las ilusiones estadísticas. Si el Lift es 1, no hay ninguna "magia" entre el Frappe y el Muffin; si es mayor a 1, descubrimos una verdadera asociación que justifica armar un combo promocional.

In [5]:
# 1. Necesitamos el Soporte aislado del consecuente (Muffin)
frecuencia_muffin = len(df_tickets[df_tickets['Muffin'] == True])
soporte_muffin = frecuencia_muffin / total_tickets

# 2. Dividimos la Confianza de la regla entre el Soporte del consecuente
lift_regla = confianza_regla / soporte_muffin

print(f"Soporte aislado del Muffin: {soporte_muffin}")
print(f"Lift (Frappe -> Muffin): {confianza_regla:.2f} / {soporte_muffin} = {lift_regla:.2f}")

if lift_regla > 1:
    print("Interpretación: LIFT > 1. El Frappe IMPULSA la venta del Muffin. Es un buen combo.")
elif lift_regla == 1:
    print("Interpretación: LIFT = 1. Coincidencia estadística. Se venden de forma independiente.")
else:
    print("Interpretación: LIFT < 1. Asociación negativa. Se evitan mutuamente.")

Soporte aislado del Muffin: 0.4
Lift (Frappe -> Muffin): 0.67 / 0.4 = 1.67
Interpretación: LIFT > 1. El Frappe IMPULSA la venta del Muffin. Es un buen combo.


In [ ]:
# Instalar librería necesaria si no la tienen: !pip install mlxtend
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. EL OLTP: Listas de artículos (Tickets de compra crudos de un OXXO)
tickets_oxxo = [
    ['Leche', 'Pan', 'Huevos'],
    ['Cerveza', 'Pañales', 'Papas'],
    ['Leche', 'Pan', 'Cerveza', 'Pañales'],
    ['Pan', 'Huevos'],
    ['Cerveza', 'Pañales', 'Refresco']
]

# 2. TRANSFORMACIÓN: ¿Recuerdan el One-Hot Encoding de la semana pasada? 
# Las reglas de asociación requieren transformar listas en matrices booleanas.
te = TransactionEncoder()
matriz_bool = te.fit_transform(tickets_oxxo)
df_transacciones = pd.DataFrame(matriz_bool, columns=te.columns_)

print("--- MATRIZ TRANSACCIONAL (El puente hacia el algoritmo) ---")
display(df_transacciones)

# 3. EXTRACCIÓN DE REGLAS (Algoritmo Apriori)
# Paso A: Encontrar combinaciones frecuentes (mínimo en el 40% de los tickets)
itemsets_frecuentes = apriori(df_transacciones, min_support=0.4, use_colnames=True)

# Paso B: Extraer las reglas de asociación
reglas = association_rules(itemsets_frecuentes, metric="lift", min_threshold=1.0)

print("\n--- REGLAS DE ASOCIACIÓN EXTRAÍDAS ---")
# Filtramos para ver solo las columnas más importantes
display(reglas[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

--- MATRIZ TRANSACCIONAL (El puente hacia el algoritmo) ---


,Cerveza,Huevos,Leche,Pan,Papas,Pañales,Refresco
0,False,True,True,True,False,False,False
1,True,False,False,False,True,True,False
2,True,False,True,True,False,True,False
3,False,True,False,True,False,False,False
4,True,False,False,False,False,True,True



--- REGLAS DE ASOCIACIÓN EXTRAÍDAS ---


,antecedents,consequents,support,confidence,lift
0,frozenset({Cerveza}),frozenset({Pañales}),0.6,1.000000,1.666667
1,frozenset({Pañales}),frozenset({Cerveza}),0.6,1.000000,1.666667
2,frozenset({Huevos}),frozenset({Pan}),0.4,1.000000,1.666667
3,frozenset({Pan}),frozenset({Huevos}),0.4,0.666667,1.666667
4,frozenset({Leche}),frozenset({Pan}),0.4,1.000000,1.666667
5,frozenset({Pan}),frozenset({Leche}),0.4,0.666667,1.666667
